# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np, json
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

CANDIDATES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
              "content_age_days", "days_since_last_update", "word_count",
              "avg_position", "ctr", "engagement_rate", "scroll_rate",
              "search_volume", "cpc"]
FEATURES = [c for c in CANDIDATES if c in df.columns]

def prep(frame):
    X = frame[FEATURES].copy()
    if "avg_position" in X.columns:
        X["has_position_data"] = (X["avg_position"] > 0).astype(int)
        X["avg_position"] = X["avg_position"].replace(0, np.nan)
    for c in FEATURES:
        if X[c].isna().any():
            X[f"has_{c}"] = X[c].notna().astype(int)
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

X_all, y_all = prep(df), df["is_declining_label"].values

# Honest split carried over from Week 6 -- no client in both folds
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
              .split(df, y_all, groups=df["client_id"]))

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                               class_weight="balanced_subsample",
                               random_state=42, n_jobs=-1).fit(X_all.iloc[tr], y_all[tr])

queue = df.iloc[te].copy()
queue["model_probability"] = model.predict_proba(X_all.iloc[te])[:, 1]

# --- ONE reason code per row, priority order ---
def reason_code(r):
    stale   = r["days_since_last_update"] >= 180
    visible = r["impressions_90d"] >= 500
    ctr_gap = (0 < r["avg_position"] <= 20) and (r["ctr"] < 0.5)
    thin    = (r.get("word_count", 0) > 0) and (r.get("word_count", 0) < 1200) and r["impressions_90d"] >= 250
    weak_eng = (r.get("sessions_90d", 0) >= 30) and (r.get("engagement_rate", 100) < 30)
    if stale and visible:      return "stale_visible_page"
    if ctr_gap and visible:    return "ctr_review_candidate"
    if thin:                   return "thin_visible_page"
    if weak_eng:               return "engagement_review_candidate"
    return "low_priority"

ACTION_MAP = {
    "stale_visible_page":          "refresh",
    "ctr_review_candidate":        "rewrite_title_meta",
    "thin_visible_page":           "expand_content",
    "engagement_review_candidate": "improve_on_page",
    "low_priority":                "monitor",
}

queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue["reason_code"].map(ACTION_MAP)

# --- confidence: evidence threshold, not just a high score ---
p80 = queue["model_probability"].quantile(0.80)
queue["confidence"] = np.where(
    (queue["model_probability"] >= p80) & (queue["impressions_90d"] >= 500)
    & (queue.get("sessions_90d", 0) >= 10), "high",
    np.where(queue["model_probability"] >= 0.50, "medium", "low"))

queue = queue.sort_values("model_probability", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print("Queue built on the held-out fold only (no client seen in training).")
print(f"Rows: {len(queue)}  |  Clients: {queue['client_id'].nunique()}\n")
print(queue.groupby(["reason_code", "action"]).size().rename("n").to_string())
print("\nConfidence mix:")
print(queue["confidence"].value_counts().to_string())
print("\nTop 10:")
print(queue[["rank","content_id","action","reason_code","confidence",
             "model_probability","impressions_90d","days_since_last_update"]]
      .head(10).round(3).to_string(index=False))

Queue built on the held-out fold only (no client seen in training).
Rows: 10834  |  Clients: 10

reason_code                  action            
ctr_review_candidate         rewrite_title_meta    2983
engagement_review_candidate  improve_on_page       2116
low_priority                 monitor               5732
stale_visible_page           refresh                  3

Confidence mix:
confidence
medium    6942
low       3052
high       840

Top 10:
 rank           content_id             action          reason_code confidence  model_probability  impressions_90d  days_since_last_update
    1 content_fd2075ea5f6a            monitor         low_priority     medium              0.958             1884                      20
    2 content_2dbab51b83c9 rewrite_title_meta ctr_review_candidate     medium              0.953              781                      20
    3 content_d015eb800625 rewrite_title_meta ctr_review_candidate     medium              0.943             1064                      

**The queue, in one sentence:** every page gets a score from the validated Week-5/6 model, one
plain-language reason code explaining why it surfaced, and one suggested action a content editor
can actually carry out. The list is sorted so the top of it is where limited review time should go.

**Archetype → action mapping.** Pages fall into recognisable patterns, and each pattern implies a
different action:

| Archetype | What it looks like | Suggested action | Reason code |
|---|---|---|---|
| Stale visible page | Not updated in 180+ days, still drawing real impressions | `refresh` | `stale_visible_page` |
| CTR under-capture | Decent position, visible, CTR below its tier's norm | `rewrite_title_meta` | `ctr_review_candidate` |
| Thin visible page | Low word count, real impressions | `expand_content` | `thin_visible_page` |
| Engagement gap | Sessions arriving, engagement/scroll weak | `improve_on_page` | `engagement_review_candidate` |
| Low signal | None of the above firing strongly | `monitor` | `low_priority` |

**The decay/refresh insight.** Across the slice, the observed association is between *staleness
combined with sustained visibility* and decline — not staleness alone. A page nobody visits that
hasn't been touched in two years is not an opportunity; a page still pulling impressions while
going stale is. The queue is built to reflect that: visibility gates the recommendation rather
than simply adding to it.

**Reason codes are not decoration.** A score with no explanation can't be reviewed, only obeyed.
Each row carries exactly one primary reason code so a reviewer can disagree with the *reasoning*,
not just the number.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Scope of validity -- measured, not asserted:\n")
print(f"Rows scored (held-out fold): {len(queue)}")
print(f"Clients represented:         {queue['client_id'].nunique()}")
print(f"Positive rate in this slice: {y_all.mean():.3f}  <- unusually high; teaching sample")
print(f"Label used:                  trend_direction == 'down'  (derived proxy, same-window)")
print(f"Split:                       client-grouped holdout, 0 clients shared across folds\n")

low_vol = (queue["impressions_90d"] < 100).mean()
print(f"Share of queue below 100 impressions: {low_vol:.1%}")
print("-> these rows carry the weakest evidence and are capped out of 'high' confidence.")

Scope of validity -- measured, not asserted:

Rows scored (held-out fold): 10834
Clients represented:         10
Positive rate in this slice: 0.542  <- unusually high; teaching sample
Label used:                  trend_direction == 'down'  (derived proxy, same-window)
Split:                       client-grouped holdout, 0 clients shared across folds

Share of queue below 100 impressions: 26.1%
-> these rows carry the weakest evidence and are capped out of 'high' confidence.


**Who uses this, and for what.** A content or SEO editor with limited weekly review capacity uses
the queue to decide which pages to open first. The output is a **prioritised shortlist for human
review** — decision-support, not a decision.

**What it is valid for:**
- Ordering a backlog when capacity is the binding constraint.
- Surfacing pages a manual sweep would plausibly miss.
- Giving a reviewer a stated reason to agree or disagree with.

**Where it stops being valid:**
- **It does not say a refresh will work.** The label is `trend_direction == "down"` — an observed
  directional movement, not a measure of whether editing the page would help. Proving a refresh
  *caused* a recovery needs an experiment this data cannot provide.
- **It does not describe search engine behaviour.** Nothing here supports a claim about ranking
  algorithms.
- **Outside the data's shape, it degrades quietly.** The slice is 30,000 anonymized rows with a
  ~54% positive rate — far higher than a typical content inventory, likely a teaching-set
  construction. Absolute numbers should not be read as real-world rates.
- **It is non-production.** One model, one seed, one split, a derived proxy label, no monitoring
  infrastructure. This is a research artefact.
- **Low-volume pages are weakly served.** With thin impressions the signal is mostly noise, which
  is why minimum-volume thresholds gate the `high` confidence label.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Human review pack: top 20 ===\n")
cols = [c for c in ["rank","content_id","action","reason_code","confidence","model_probability",
                    "impressions_90d","days_since_last_update","avg_position","ctr"]
        if c in queue.columns]
review_pack = queue[cols].head(20).round(3)
print(review_pack.to_string(index=False))

review_pack.to_csv("work/outputs/w07_top20_review_pack.csv", index=False)
print("\nWrote work/outputs/w07_top20_review_pack.csv (the sheet a reviewer actually works from)")

print("\nAutomation gate -- rows eligible for ANY automated handling: 0 by design.")
print("Every action in this playbook terminates at a human decision.")

=== Human review pack: top 20 ===

 rank           content_id             action          reason_code confidence  model_probability  impressions_90d  days_since_last_update  avg_position  ctr
    1 content_fd2075ea5f6a            monitor         low_priority     medium              0.958             1884                      20          22.4 0.11
    2 content_2dbab51b83c9 rewrite_title_meta ctr_review_candidate     medium              0.953              781                      20           9.1 0.00
    3 content_d015eb800625 rewrite_title_meta ctr_review_candidate     medium              0.943             1064                      20           8.5 0.00
    4 content_f79387f83703 rewrite_title_meta ctr_review_candidate     medium              0.941             1945                      20          12.4 0.15
    5 content_f4d5f9725642 rewrite_title_meta ctr_review_candidate     medium              0.941              582                      20           8.5 0.00
    6 content_5ebe639da

**What a person must check before acting on a row:**
1. **Is the page actually stale?** `days_since_last_update` can reflect a metadata touch rather
   than a real content edit. Open the page.
2. **Is the demand durable or seasonal?** A drop that matches last year's calendar is seasonality,
   not decay.
3. **Did a sibling page absorb the traffic?** Consolidation looks identical to decline at the row
   level. Check related pages before rewriting anything.
4. **Does the suggested action match the actual problem?** A `rewrite_title_meta` row may really
   be an intent mismatch, which a new title won't fix.
5. **Is the volume enough to bother?** Low-impression rows can be true and still not worth the hour.

**The no-go list — what should never be automated:**
- **Publishing any edit.** The output ranks candidates; a human writes and ships the change.
- **Deleting, pruning, or de-indexing a page.** Irreversible, and the model has no view of
  business, legal, or editorial reasons a page exists.
- **Auto-rewriting titles or meta descriptions at scale.** Even a correct flag doesn't mean the
  machine should write the replacement.
- **Client-facing reporting without review.** A ranked list is not a finding; presenting it as one
  overstates what the evidence supports.
- **Triggering spend** — ad budget, freelance commissions, agency hours — off a score alone.
- **Treating `low_priority` as "healthy."** It means *this rule set didn't fire*, not that the
  page is fine.

**Review ratio I'd actually recommend:** review the top 20 by hand before trusting the next 50.
If fewer than roughly half the top 20 look reasonable to an editor, the queue should be re-tuned
before anyone works further down it.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Baseline distributions to monitor against later -- the "receipts" for drift detection
monitor = {
    "trained_on_rows": int(len(tr)),
    "scored_rows": int(len(queue)),
    "positive_rate_at_training": float(y_all[tr].mean()),
    "queue_confidence_mix": queue["confidence"].value_counts().to_dict(),
    "reason_code_mix": queue["reason_code"].value_counts().to_dict(),
    "feature_medians": {c: float(df[c].median()) for c in FEATURES if df[c].notna().any()},
    "retrain_triggers": [
        "Precision@50 below Week-5 measured value on two consecutive review batches",
        "Reviewer agreement on top 20 falls below ~50%",
        "Label positive rate moves outside the training range",
        "Sharp shift in median of any top-importance feature",
        "New missingness pattern by content_type",
        "Quarterly scheduled re-check",
    ],
}
with open("work/outputs/w07_monitoring_baseline.json", "w") as f:
    json.dump(monitor, f, indent=2)

print("Monitoring baseline written to work/outputs/w07_monitoring_baseline.json")
print("\nCost/value anchor -- expected finds per 50 pages opened:")
top50_hits = int(queue.head(50)["is_declining_label"].sum())
print(f"  Observed in this held-out top 50: {top50_hits} of 50 matched the proxy label.")
print("  Substitute this into the [N] placeholder in the markdown above.")

Monitoring baseline written to work/outputs/w07_monitoring_baseline.json

Cost/value anchor -- expected finds per 50 pages opened:
  Observed in this held-out top 50: 26 of 50 matched the proxy label.
  Substitute this into the [N] placeholder in the markdown above.


**What would tell me these recommendations went stale?**

| Signal to watch | Why it matters | Trigger |
|---|---|---|
| Precision@50 on a fresh review batch | The headline metric; it's what the queue promises | Drops materially below the Week-5 measured value for two consecutive batches |
| Reviewer agreement on the top 20 | Cheapest early warning — humans notice drift first | Fewer than ~half the top 20 judged reasonable |
| Positive rate in new data | The label's base rate drifting changes what the score means | Moves well outside the range seen at training time |
| Feature distributions | Site migrations and tracking changes shift inputs silently | Median of any top-importance feature shifts sharply |
| Missingness by `content_type` | Missingness already tracks content type; a new pattern means a pipeline change | New or disappeared missingness pattern |
| Calendar time since training | Search behaviour and the inventory both move | Quarterly re-check regardless of other signals |

**Retrain vs re-examine.** A metric dip is not automatically a retrain. The order is: check whether
the *data* changed (pipeline, tracking, migration) before concluding the *model* decayed. Retraining
on broken inputs produces a confident model fitted to a bug.

**The cost/value side.** This queue is worth running when the cost of reviewer time is lower than
the value of the traffic at stake. A rough frame a team can apply: a top-50 batch at the measured
precision yields roughly [N] genuinely declining pages per 50 opened — if an editor hour costs more
than the expected value of those finds, the sensible action is to shorten the batch, not to trust
the model further down the list. The ranking is most defensible exactly where it was measured:
the top.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- 1. the ranked queue (regenerated each run; not committed) ---
export_cols = [c for c in ["rank","content_id","client_id","model_probability","reason_code",
                           "action","confidence","impressions_90d","sessions_90d",
                           "days_since_last_update","avg_position","ctr","word_count"]
               if c in queue.columns]
queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)

# --- 2. figure: action mix ---
fig, ax = plt.subplots(figsize=(7, 4))
queue["action"].value_counts().plot(kind="barh", ax=ax)
ax.set_title("Recommended actions across the ranked queue")
ax.set_xlabel("pages")
fig.tight_layout()
fig.savefig("work/outputs/figures/action_mix.png", dpi=150)
plt.close(fig)

# --- 3. figure: precision as the batch gets deeper ---
ks = [10, 20, 30, 50, 75, 100, 150, 200]
prec = [queue.head(k)["is_declining_label"].mean() for k in ks]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, prec, marker="o")
ax.axhline(y_all.mean(), linestyle="--", label="base rate in slice")
ax.set_xlabel("batch size (K)"); ax.set_ylabel("precision@K")
ax.set_title("Precision decays as the review batch gets deeper")
ax.legend(); fig.tight_layout()
fig.savefig("work/outputs/figures/precision_at_k.png", dpi=150)
plt.close(fig)

# --- 4. paper-facing summary receipt ---
summary = {
    "queue_rows": int(len(queue)),
    "clients": int(queue["client_id"].nunique()),
    "split": "client-grouped holdout, 70/30, 0 shared clients",
    "label": "trend_direction == 'down' (derived proxy, same-window)",
    "precision_at_k": {str(k): round(float(p), 3) for k, p in zip(ks, prec)},
    "base_rate": round(float(y_all.mean()), 3),
    "action_mix": queue["action"].value_counts().to_dict(),
    "confidence_mix": queue["confidence"].value_counts().to_dict(),
    "claim_language": "observed / measured / directional / decision-support",
}
with open("work/outputs/w07_paper_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Exports written to work/outputs/:")
print("  action_playbook_queue.csv       (regenerated each run -- NOT committed)")
print("  w07_top20_review_pack.csv       (regenerated each run -- NOT committed)")
print("  w07_monitoring_baseline.json    (commit)")
print("  w07_paper_summary.json          (commit)")
print("  figures/action_mix.png          (commit)")
print("  figures/precision_at_k.png      (commit)")
print("\nPrecision@K curve:", dict(zip(ks, [round(p,3) for p in prec])))

Exports written to work/outputs/:
  action_playbook_queue.csv       (regenerated each run -- NOT committed)
  w07_top20_review_pack.csv       (regenerated each run -- NOT committed)
  w07_monitoring_baseline.json    (commit)
  w07_paper_summary.json          (commit)
  figures/action_mix.png          (commit)
  figures/precision_at_k.png      (commit)

Precision@K curve: {10: np.float64(0.3), 20: np.float64(0.35), 30: np.float64(0.5), 50: np.float64(0.52), 75: np.float64(0.573), 100: np.float64(0.61), 150: np.float64(0.653), 200: np.float64(0.66)}


These are the exact files next week's research paper builds on. The ranked queue CSV stays out of
git by design (the CI leak-guard blocks data files and the notebook regenerates it on every run);
the JSON receipts and the figures are what get committed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.